In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [4]:
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("balance", DoubleType(), True),
    StructField("vip_status", StringType(), True)
])

In [6]:
card_trn_schema = StructType([
    StructField("trn_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("trn_date", StringType(), True),
    StructField("merchant", StringType(), True)
])

In [15]:
df_customers = (
    spark.read
    .option("header", "true")
    .schema(customer_schema)
    .csv("s3a://spark/customers.csv")
)

df_card_trn = (
    spark.read
    .option("header", "true")
    .schema(card_trn_schema)
    .csv("s3a://spark/card_trn.csv")
)

In [16]:
df_card_trn = df_card_trn.withColumn("trn_date", F.to_date(F.col("trn_date"), "yyyy-MM-dd"))

In [18]:
df_cust_transformed = (
    df_customers
    .withColumn(
        "balance", F.when(F.col("balance").isNull(), 0.0).otherwise(F.col("balance")))
    .withColumn(
        "age_group", F.when(F.col("age") < 25, "young")
         .when((F.col("age") >= 25) & (F.col("age") <= 40), "adult")
         .otherwise("senior"))
    .withColumnRenamed("vip_status", "flg_is_vip")
    .withColumn("insert_date", F.current_date())
)



In [24]:
df_cust_transformed.show(5, truncate=False)

+-----------+---------------+---+-------+----------+---------+-----------+
|customer_id|customer_name  |age|balance|flg_is_vip|age_group|insert_date|
+-----------+---------------+---+-------+----------+---------+-----------+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |young    |2026-09-11 |
|C002       |Sabina Mammadov|60 |3651.47|Y         |senior   |2026-09-11 |
|C003       |Sevinj Karimova|19 |0.0    |Y         |young    |2026-09-11 |
|C004       |Nigar Guliyev  |27 |2427.85|Y         |adult    |2026-09-11 |
|C005       |Vusal Guliyev  |23 |3179.4 |N         |young    |2026-09-11 |
+-----------+---------------+---+-------+----------+---------+-----------+
only showing top 5 rows



In [20]:
(
    df_cust_transformed.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .parquet("s3a://spark/silver/customers")
)

In [21]:
df_trn = (
    df_card_trn
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total_amount"))
)

In [26]:
df_trn.show(5, truncate=False)

+-----------+------------------+
|customer_id|total_amount      |
+-----------+------------------+
|C006       |166.53            |
|C010       |251.5             |
|C007       |442.89            |
|C012       |641.71            |
|C003       |1471.8700000000001|
+-----------+------------------+
only showing top 5 rows



In [23]:
df_result = (
    df_cust_transformed
    .join(df_trn, on="customer_id", how="inner")
    .select("customer_name", "total_amount")
    .orderBy(F.col("total_amount").desc())
)

df_result.show(2, truncate=False)

+---------------+------------------+
|customer_name  |total_amount      |
+---------------+------------------+
|Sevinj Karimova|1471.8700000000001|
|Gunel Mammadov |1471.15           |
+---------------+------------------+
only showing top 2 rows

